# MULTIMAMBA — Multimodal Hb Estimation (Eye Image + Weight / Height / SES)

Extends every image-only Mamba / CNN / ViT model from `run_ensemble.ipynb` into a
**multimodal** model that fuses the eye image with tabular clinical variables
(**Height, Weight, Socio-economic status = APL/BPL**).

**How it works.** A single generic `MultiModalWrapper` wraps *any* existing model
(all of them end in `.cls_head` / `.reg_head`). A forward-pre-hook grabs the pooled
image feature going into `cls_head`, fuses it with an encoding of the tabular
variables, and feeds the result to fresh dual heads. Nothing in the original model
files is modified.

**Fusion modes compared** (literature shows plain concat under-mixes modalities —
FiLM / gating / attention fuse better): `concat`, `film`, `gate`, `attn`.

**Structure**
1. Config — *edit only the first cell* (K for CV, task, columns, tabular toggles, fusion).
2. Data + tabular preprocessing (per-fold standardisation → no leakage).
3. Fusion wrapper + task-specific losses/metrics.
4. K-fold runner (K=1 → single holdout; K≥2 → StratifiedKFold, test mean±std).
5. **Main pass** — all models with the default fusion.
6. **Ablations (end of file)** — fusion-mode sweep, per-variable contribution
   (image-only → +Height → +Weight → +SES), and a tabular-only reference.


In [ ]:
# ==============================================================
#            ALL VARIABLES -- EDIT ONLY THIS BLOCK
# ==============================================================

# ---- Cross-validation --------------------------------------------------------
K = 5                       # 1..5 : K=1 -> single holdout, K>=2 -> StratifiedKFold
assert 1 <= K <= 5, "K must be between 1 and 5"

# ---- Task --------------------------------------------------------------------
TASK = "classification"     # "classification" (anemic < HB_THRESH) | "regression" (exact Hb)

# ---- Dataset paths -----------------------------------------------------------
IMAGE_DIR   = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye"
CSV_PATH    = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx"
IMAGE_DIR_2 = ""            # optional 2nd dataset slot (e.g. right_eye)
CSV_PATH_2  = ""

# ---- Column names (must match the Excel headers EXACTLY) ---------------------
IMAGE_COL  = "Patient ID"
HB_COL     = "Haemoglobin (gm/dL)"
HB_THRESH  = 12.0
HEIGHT_COL = "Height"
WEIGHT_COL = "Weight"
SES_COL    = "Socio-economic status"
SES_MAP    = {"APL": 0.0, "BPL": 1.0}     # APL/BPL -> numeric (unknown -> 0.0)

# ---- Multimodal / tabular ----------------------------------------------------
USE_TABULAR      = True                    # master switch: add tabular vars to the model
USE_SES          = True                    # include socio-economic status feature
NUMERIC_FEATURES = [HEIGHT_COL, WEIGHT_COL]# standardised numeric tabular features
FUSION           = "film"                  # MAIN-pass fusion: "concat"|"film"|"gate"|"attn"
FUSION_MODES     = ["concat", "film", "gate", "attn"]   # compared in the ablation
TAB_HIDDEN       = 64                       # tabular MLP width
FUSE_DIM         = 128                      # token dim for "attn" fusion
TAB_DROPOUT      = 0.2

# ---- HB range filter / augmentation -----------------------------------------
HB_FILTER_MIN = 7.0
HB_FILTER_MAX = 15.0
USE_AUGMENTATION = True
AUG_BINS         = 10

# ---- Training ----------------------------------------------------------------
EPOCHS      = 30
BATCH_SIZE  = 32
LR          = 1e-4
VAL_SPLIT   = 0.2          # used when K == 1
TEST_SPLIT  = 0.1          # fixed held-out test set (locked across all folds)
NUM_WORKERS = 4
SEED        = 42
EARLY_STOP  = 8
SCHEDULER   = "cosine"     # "cosine" | "plateau" | None
GRAD_ACCUM  = 1
WARMUP_EPOCHS = 3
USE_AMP     = True
FREEZE_BACKBONE = True     # EfficientMamba / timm baselines
IMG_SIZE    = 224

# ---- Preprocessing -----------------------------------------------------------
PREPROCESS = {"colorspace": "RGB", "use_clahe": False,
              "clahe_clip": 2.0, "clahe_grid": (8, 8)}

# ---- Classification-specific settings ---------------------------------------
CLS_CONFIG = {"use_focal_loss": True, "focal_gamma": 2.0,
              "use_class_weights": True, "threshold": 0.5}

# ---- Regression-specific settings -------------------------------------------
REG_CONFIG = {"normalize_hb": True, "hb_min": 4.0, "hb_max": 20.0,
              "loss_fn": "huber", "huber_delta": 1.0}

# ---- Novel-model architecture cfg -------------------------------------------
NEW_MODEL_CFG = {"embed_dim": 128, "depth": 4, "d_state": 64,
                 "colour_proj": "learned", "n_experts": 5}

# ---- Which models to run in the MAIN pass -----------------------------------
RUN = {
    # Mamba variants
    "Mamba1": True, "Mamba2": True, "Mamba3": True, "VMamba": True,
    "MambaVision": True, "MedMamba": True, "VSSD": True, "DSA_Mamba": True,
    "EfficientMamba": True, "AdaptiveScan": True, "MoEMamba": True, "CrossFusion": True,
    # CNN / ViT baselines
    "ResNet50": True, "EfficientNetB3": True, "DenseNet121": True,
    "MobileNetV3": True, "ViT_B16": True,
}

# ---- Ablation config (runs at the END of the notebook) ----------------------
ABLATION_MODEL        = "Mamba2"   # backbone used for the ablations (any key in RUN)
RUN_ABLATION_FUSION   = True       # compare every mode in FUSION_MODES
RUN_ABLATION_VARIABLE = True       # image-only -> +Height -> +Weight -> +SES
RUN_ABLATION_TABONLY  = True       # tabular-only MLP reference (no image)

# Each step = (label, numeric_feature_list, use_ses).  Shows each variable's marginal gain.
ABLATION_STEPS = [
    ("image-only",           [],                       False),
    ("+Height",              [HEIGHT_COL],             False),
    ("+Height+Weight",       [HEIGHT_COL, WEIGHT_COL], False),
    ("+Height+Weight+SES",   [HEIGHT_COL, WEIGHT_COL], True ),
]

# ---- Repo path autodetect ----------------------------------------------------
import os, sys
def _find_repo_root():
    cands = [os.getcwd(), os.path.join(os.getcwd(), "ALLMAMBA")]
    try:
        for d in os.listdir(os.getcwd()):
            p = os.path.join(os.getcwd(), d)
            if os.path.isdir(p): cands.append(p)
    except Exception: pass
    cands.append("/kaggle/working/ALLMAMBA")
    for c in cands:
        if os.path.isdir(os.path.join(c, "MAMBA_MODELS")): return c
    return os.getcwd()

REPO_ROOT  = _find_repo_root()
MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
if MODELS_DIR not in sys.path: sys.path.insert(0, MODELS_DIR)

CV_DESC = "single holdout" if K == 1 else f"{K}-fold StratifiedKFold"
print(f"TASK        : {TASK}")
print(f"Cross-val   : K={K}  ({CV_DESC})")
print(f"Multimodal  : USE_TABULAR={USE_TABULAR}  USE_SES={USE_SES}  fusion={FUSION}")
print(f"Numeric feat: {NUMERIC_FEATURES}")
print(f"MODELS_DIR  : {MODELS_DIR}")
print(f"Models ON   : {[k for k,v in RUN.items() if v]}")


## Section 2 — Install dependencies (Kaggle)

Run once on a fresh machine. Safe to skip if the environment already has the packages.

In [ ]:
# ---- Install deps (Kaggle). Comment out if already installed. ----
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
# _pip("mamba-ssm", "causal-conv1d", "einops", "timm", "pandas", "openpyxl",
#      "scikit-learn", "tqdm", "matplotlib", "opencv-python")
print("If packages are missing, uncomment the _pip(...) line above and re-run.")


## Step 0 — Clone repo (Kaggle only; skip if already present)

In [ ]:
import subprocess, os
# REPO_URL = "https://github.com/YOUR_USERNAME/MULTIMAMBA.git"
# if not os.path.isdir("ALLMAMBA/MAMBA_MODELS") and not os.path.isdir("MAMBA_MODELS"):
#     subprocess.run(["git", "clone", REPO_URL, "ALLMAMBA"], check=False)
print("Repo present:", os.path.isdir(MODELS_DIR))


## Section 3 — Imports & device

In [ ]:
import os, sys, math, time, warnings, traceback, importlib.util, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report, roc_auc_score,
                             mean_absolute_error, f1_score, confusion_matrix, r2_score)
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}   PyTorch: {torch.__version__}")
if MODELS_DIR not in sys.path: sys.path.insert(0, MODELS_DIR)


## Section 4 — Data: merge, filter, and lock the fixed test set

Reuses the exact `merge_and_filter_datasets` loader from `common/augment.py` (it
preserves every Excel column, so Height/Weight/SES flow through untouched). We build
the fixed 10% held-out **test set** and the **train+val pool**; folds are drawn from
the pool only. Tabular columns are validated and their coverage reported.

In [ ]:
from common.augment import merge_and_filter_datasets, make_balanced_loader

# ---------- image preprocessing (same as run_ensemble) ----------
class CLAHETransform:
    def __call__(self, img):
        import cv2
        lab = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=PREPROCESS["clahe_clip"],
                                tileGridSize=PREPROCESS["clahe_grid"])
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

class ToLABTensor:
    def __call__(self, img):
        import cv2
        lab = cv2.cvtColor(np.array(img).astype(np.uint8), cv2.COLOR_RGB2LAB).astype(np.float32)
        lab[:, :, 0] /= 255.0
        lab[:, :, 1] = (lab[:, :, 1] - 128.0) / 127.0
        lab[:, :, 2] = (lab[:, :, 2] - 128.0) / 127.0
        return torch.from_numpy(lab.transpose(2, 0, 1))

class ToHSVTensor:
    def __call__(self, img):
        import cv2
        hsv = cv2.cvtColor(np.array(img).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] /= 179.0; hsv[:, :, 1] /= 255.0; hsv[:, :, 2] /= 255.0
        return torch.from_numpy(hsv.transpose(2, 0, 1))

def build_transforms(augment):
    steps = []
    if augment:
        steps += [transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
                  transforms.RandomCrop(IMG_SIZE),
                  transforms.RandomHorizontalFlip(),
                  transforms.RandomRotation(15),
                  transforms.ColorJitter(0.25, 0.25, 0.15)]
    else:
        steps.append(transforms.Resize((IMG_SIZE, IMG_SIZE)))
    if PREPROCESS["use_clahe"]:
        steps.append(CLAHETransform())
    cs = PREPROCESS["colorspace"].upper()
    if cs == "RGB":
        steps += [transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)]
    elif cs == "CIELAB": steps.append(ToLABTensor())
    elif cs == "HSV":    steps.append(ToHSVTensor())
    elif cs in ("GRAY", "GRAYSCALE"):
        steps += [transforms.Grayscale(3), transforms.ToTensor(),
                  transforms.Normalize((0.5,)*3, (0.5,)*3)]
    else: raise ValueError(f"Unknown colorspace {PREPROCESS['colorspace']}")
    return transforms.Compose(steps)

_IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ""]
def find_image_path(pid, img_dir):
    for ext in _IMG_EXTS:
        p = os.path.join(img_dir, str(pid) + ext)
        if os.path.exists(p): return p
    return None

# ---------- load + merge ----------
def _load_full_df():
    slot_dfs, slot_dirs = [], []
    for csv_p, img_d in [(CSV_PATH, IMAGE_DIR), (CSV_PATH_2, IMAGE_DIR_2)]:
        if csv_p and img_d:
            _df = (pd.read_excel(csv_p) if csv_p.endswith((".xlsx", ".xls"))
                   else pd.read_csv(csv_p))
            slot_dfs.append(_df); slot_dirs.append(img_d)
    if not slot_dfs:
        raise RuntimeError("No dataset slots active. Set CSV_PATH and IMAGE_DIR.")
    df = merge_and_filter_datasets(
        dataframes=slot_dfs, image_dirs=slot_dirs, hb_col=HB_COL, image_col=IMAGE_COL,
        hb_filter_min=HB_FILTER_MIN, hb_filter_max=HB_FILTER_MAX,
        find_image_path_fn=find_image_path, verbose=True)
    if len(df) == 0:
        raise RuntimeError("Zero valid samples after filtering.")
    return df.reset_index(drop=True)

FULL = _load_full_df()

# ---------- validate tabular columns ----------
def _report_col(col):
    if col not in FULL.columns:
        print(f"  [WARN] tabular column '{col}' NOT found in the sheet -> "
              f"it will be imputed as 0. Available: {list(FULL.columns)}")
        return False
    if col == SES_COL:
        n_ok = FULL[col].astype(str).str.strip().isin(list(SES_MAP)).sum()
    else:
        n_ok = pd.to_numeric(FULL[col], errors='coerce').notna().sum()
    print(f"  [OK]  '{col}': {n_ok}/{len(FULL)} usable values")
    return True

print("Tabular column check:")
for c in NUMERIC_FEATURES: _report_col(c)
_report_col(SES_COL)
if SES_COL in FULL.columns:
    print("  SES value counts:", FULL[SES_COL].astype(str).str.strip().value_counts().to_dict())

# ---------- fixed test split + train/val pool ----------
_binary = (FULL[HB_COL] < HB_THRESH).astype(int)
POOL_DF, TEST_DF = train_test_split(
    FULL, test_size=TEST_SPLIT, random_state=SEED, stratify=_binary)
POOL_DF  = POOL_DF.reset_index(drop=True)
TEST_DF  = TEST_DF.reset_index(drop=True)
POOL_LBL = (POOL_DF[HB_COL] < HB_THRESH).astype(int).values

n_anem = int(_binary.sum())
print(f"\n  Samples: {len(FULL)}  | Anemic(<{HB_THRESH}): {n_anem} "
      f"({n_anem/len(FULL)*100:.1f}%)  Normal: {len(FULL)-n_anem}")
print(f"  Train+Val pool: {len(POOL_DF)}   Fixed Test: {len(TEST_DF)}")
print(f"  Hb -> mean={FULL[HB_COL].mean():.2f}  std={FULL[HB_COL].std():.2f}  "
      f"min={FULL[HB_COL].min():.2f}  max={FULL[HB_COL].max():.2f}")


## Section 5 — Tabular preprocessing & multimodal dataset

`fit_tab_stats` computes mean/std of the numeric variables **on training rows only**
(per fold) so the test/val standardisation never leaks. `MMDataset` returns
`(image, tabular_vector, class_label, hb_value)`; the tabular vector length adapts to
the active feature set (`numeric_feats` + optional SES).

In [ ]:
def fit_tab_stats(df, numeric_feats):
    """Per-fold standardisation stats from TRAIN rows only. Returns {col:(mean,std)}."""
    stats = {}
    for c in numeric_feats:
        if c in df.columns:
            v = pd.to_numeric(df[c], errors="coerce")
            m = float(v.mean()) if v.notna().any() else 0.0
            s = float(v.std())  if v.notna().any() else 1.0
        else:
            m, s = 0.0, 1.0
        stats[c] = (m, max(s, 1e-6))
    return stats

def _row_tab_vector(row, numeric_feats, use_ses, stats):
    vec = []
    for c in numeric_feats:
        m, s = stats[c]
        x = pd.to_numeric(pd.Series([row.get(c, np.nan)]), errors="coerce").iloc[0]
        vec.append(0.0 if pd.isna(x) else (float(x) - m) / s)   # nan -> mean(=0)
    if use_ses:
        raw = str(row.get(SES_COL, "")).strip()
        vec.append(float(SES_MAP.get(raw, 0.0)))
    return np.asarray(vec, dtype=np.float32)

class MMDataset(Dataset):
    def __init__(self, df, transform, numeric_feats, use_ses, stats):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.numeric_feats = numeric_feats
        self.use_ses = use_ses
        self.stats = stats
        self.tab_dim = len(numeric_feats) + (1 if use_ses else 0)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        hb  = float(row[HB_COL])
        lbl = 0 if hb < HB_THRESH else 1
        img = Image.open(row["_img_path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        tab = torch.from_numpy(_row_tab_vector(row, self.numeric_feats, self.use_ses, self.stats))
        if self.tab_dim == 0:                      # image-only ablation
            tab = torch.zeros(0, dtype=torch.float32)
        return img, tab, torch.tensor(lbl, dtype=torch.long), torch.tensor(hb, dtype=torch.float32)

print("Tabular + MMDataset ready.")


## Section 6 — Multimodal fusion wrapper (works on ANY model)

`MultiModalWrapper` wraps any `(logits, hb)` image model. A forward-pre-hook captures
the pooled feature entering `cls_head` (`feat_dim` is read straight off that head).
Four fusion modes:

- **concat** — glue `[image_feat ; tab_feat]`, then heads (simple baseline).
- **film**  — tabular data predicts per-channel `(gamma, beta)`; `feat*(1+gamma)+beta` (feature-wise modulation).
- **gate**  — tabular data predicts a sigmoid gate over image channels.
- **attn**  — image feature + each tabular scalar become tokens; a Transformer
  encoder layer lets them cross-attend; the image token output is used.

`ImageOnlyWrapper` (ignores tab) and `TabularOnlyModel` (ignores image) share the same
`forward(img, tab)` signature so the training loop is identical everywhere.

In [ ]:
def _infer_feat_dim(cls_head):
    if isinstance(cls_head, nn.Linear):
        return cls_head.in_features
    for m in cls_head.modules():
        if isinstance(m, nn.Linear):   return m.in_features
        if isinstance(m, nn.LayerNorm):
            ns = m.normalized_shape
            return ns[0] if isinstance(ns, (tuple, list)) else int(ns)
    raise ValueError("Could not infer feat_dim from cls_head")

def _mlp(inp, hid):
    return nn.Sequential(nn.Linear(inp, hid), nn.GELU(),
                         nn.LayerNorm(hid), nn.Linear(hid, hid), nn.GELU())

def _make_heads(dim, dropout):
    cls = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(dropout), nn.Linear(dim, 2))
    reg = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, max(dim // 2, 8)),
                        nn.GELU(), nn.Linear(max(dim // 2, 8), 1))
    return cls, reg

class MultiModalWrapper(nn.Module):
    def __init__(self, image_model, tab_dim, fusion="film",
                 tab_hidden=TAB_HIDDEN, fuse_dim=FUSE_DIM, dropout=TAB_DROPOUT):
        super().__init__()
        assert tab_dim > 0, "MultiModalWrapper needs tab_dim>0 (use ImageOnlyWrapper otherwise)"
        self.image_model = image_model
        self.fusion = fusion
        self.tab_dim = tab_dim
        feat_dim = _infer_feat_dim(image_model.cls_head)
        self.feat_dim = feat_dim
        self._feat = None
        image_model.cls_head.register_forward_pre_hook(self._hook)

        if fusion == "concat":
            self.tab_enc = _mlp(tab_dim, tab_hidden); fused = feat_dim + tab_hidden
        elif fusion == "film":
            self.tab_enc = _mlp(tab_dim, tab_hidden)
            self.film = nn.Linear(tab_hidden, feat_dim * 2); fused = feat_dim
        elif fusion == "gate":
            self.tab_enc = _mlp(tab_dim, tab_hidden)
            self.gate = nn.Linear(tab_hidden, feat_dim); fused = feat_dim
        elif fusion == "attn":
            self.img_proj = nn.Linear(feat_dim, fuse_dim)
            self.tok_proj = nn.ModuleList([nn.Linear(1, fuse_dim) for _ in range(tab_dim)])
            self.attn = nn.TransformerEncoderLayer(
                d_model=fuse_dim, nhead=4, dim_feedforward=fuse_dim * 2,
                dropout=dropout, batch_first=True, activation="gelu")
            fused = fuse_dim
        else:
            raise ValueError(f"Unknown fusion '{fusion}'")
        self.fused_dim = fused
        self.cls_head, self.reg_head = _make_heads(fused, dropout)

    def _hook(self, module, inp): self._feat = inp[0]

    def forward(self, img, tab):
        _ = self.image_model(img)               # runs backbone; hook captures pooled feat
        f = self._feat                           # (B, feat_dim)
        tab = tab.to(f.dtype)
        if self.fusion == "concat":
            fused = torch.cat([f, self.tab_enc(tab)], dim=-1)
        elif self.fusion == "film":
            g, b = self.film(self.tab_enc(tab)).chunk(2, dim=-1)
            fused = f * (1.0 + g) + b
        elif self.fusion == "gate":
            fused = f * torch.sigmoid(self.gate(self.tab_enc(tab)))
        elif self.fusion == "attn":
            toks = [self.img_proj(f).unsqueeze(1)]
            for i, proj in enumerate(self.tok_proj):
                toks.append(proj(tab[:, i:i+1]).unsqueeze(1))
            seq = torch.cat(toks, dim=1)         # (B, 1+tab_dim, fuse_dim)
            fused = self.attn(seq)[:, 0]         # image-token output
        return self.cls_head(fused), self.reg_head(fused).squeeze(-1)

class ImageOnlyWrapper(nn.Module):
    """Baseline: ignores tabular input, keeps the model's own heads. Unified signature."""
    def __init__(self, image_model):
        super().__init__(); self.image_model = image_model
    def forward(self, img, tab):
        return self.image_model(img)

class TabularOnlyModel(nn.Module):
    """Reference: predicts from tabular variables alone (no image)."""
    def __init__(self, tab_dim, hidden=64, dropout=TAB_DROPOUT):
        super().__init__()
        self.net = _mlp(max(tab_dim, 1), hidden)
        self.tab_dim = tab_dim
        self.cls_head, self.reg_head = _make_heads(hidden, dropout)
    def forward(self, img, tab):
        if self.tab_dim == 0:
            tab = torch.zeros(img.size(0), 1, device=img.device)
        h = self.net(tab)
        return self.cls_head(h), self.reg_head(h).squeeze(-1)

print("Fusion wrappers ready:  MultiModalWrapper / ImageOnlyWrapper / TabularOnlyModel")


## Section 7 — Task-specific losses & metrics

Classification and regression use **separate** functions — nothing shared beyond the
training skeleton. Classification: Focal/CE with balanced class weights, evaluated by
Accuracy / AUC / macro-F1 with threshold tuning. Regression: Huber/MAE/MSE on
normalised Hb, evaluated by MAE / RMSE / R2.

In [ ]:
# ---------------- CLASSIFICATION ----------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__(); self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def build_cls_loss(train_labels):
    wt = None
    if CLS_CONFIG["use_class_weights"] and len(train_labels) > 0:
        w = compute_class_weight("balanced", classes=np.array([0, 1]),
                                 y=np.array(train_labels))
        wt = torch.tensor(w, dtype=torch.float32).to(DEVICE)
    if CLS_CONFIG["use_focal_loss"]:
        return FocalLoss(gamma=CLS_CONFIG["focal_gamma"], weight=wt)
    return nn.CrossEntropyLoss(weight=wt)

def cls_metrics(labels, preds, scores):
    m = {"acc": accuracy_score(labels, preds),
         "f1":  f1_score(labels, preds, average="macro", zero_division=0)}
    try:    m["auc"] = roc_auc_score(labels, scores)
    except Exception: m["auc"] = float("nan")
    return m

def find_best_threshold(labels, scores):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.10, 0.91, 0.01):
        f = f1_score(labels, [1 if s >= t else 0 for s in scores],
                     average="macro", zero_division=0)
        if f > best_f: best_f, best_t = f, round(float(t), 2)
    return best_t, best_f

# ---------------- REGRESSION ----------------
def _norm_hb(hb):   return (hb - REG_CONFIG["hb_min"]) / (REG_CONFIG["hb_max"] - REG_CONFIG["hb_min"])
def _denorm_hb(x):  return x * (REG_CONFIG["hb_max"] - REG_CONFIG["hb_min"]) + REG_CONFIG["hb_min"]

def build_reg_loss():
    fn = REG_CONFIG["loss_fn"].lower()
    if fn == "huber": return nn.HuberLoss(delta=REG_CONFIG["huber_delta"])
    if fn == "mae":   return nn.L1Loss()
    return nn.MSELoss()

def reg_metrics(hb_true, hb_pred):
    at, ap = np.asarray(hb_true), np.asarray(hb_pred)
    return {"mae":  float(np.mean(np.abs(at - ap))),
            "rmse": float(np.sqrt(np.mean((at - ap) ** 2))),
            "r2":   float(r2_score(at, ap)) if len(at) > 1 else float("nan")}

class EarlyStopper:
    def __init__(self, patience): self.p = patience; self.n = 0; self.best = float("inf")
    def step(self, s):
        if s < self.best - 1e-6: self.best = s; self.n = 0; return False
        self.n += 1; return self.p > 0 and self.n >= self.p

print("Losses & metrics ready (classification and regression kept separate).")


## Section 8 — Multimodal epoch loop

Unpacks `(img, tab, label, hb)` and calls `model(img, tab)`. Loss dispatch is
task-specific; AMP, gradient accumulation, and clipping match the original pipeline.

In [ ]:
def run_epoch_mm(model, loader, cls_loss=None, reg_loss=None,
                 optimizer=None, grad_accum=1, scaler=None):
    import contextlib
    training = optimizer is not None
    model.train() if training else model.eval()
    total = 0.0
    P, Y, S, HBP, HBT = [], [], [], [], []
    if training: optimizer.zero_grad()
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for bi, (imgs, tab, labels, hb) in enumerate(
                tqdm(loader, leave=False, desc="train" if training else "eval ")):
            imgs, tab = imgs.to(DEVICE), tab.to(DEVICE)
            labels, hb = labels.to(DEVICE), hb.to(DEVICE)
            amp = torch.cuda.amp.autocast() if scaler is not None else contextlib.nullcontext()
            with amp:
                logits, hb_pred = model(imgs, tab)
                if TASK == "classification":
                    loss = cls_loss(logits, labels)
                else:
                    ht = _norm_hb(hb) if REG_CONFIG["normalize_hb"] else hb
                    loss = reg_loss(hb_pred.view(-1), ht.view(-1))
            if training:
                (scaler.scale(loss / grad_accum) if scaler else (loss / grad_accum)).backward()
                last = (bi + 1 == len(loader))
                if (bi + 1) % grad_accum == 0 or last:
                    if scaler:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer); scaler.update()
                    else:
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    optimizer.zero_grad()
            total += loss.item()
            if TASK == "classification":
                probs = torch.softmax(logits, dim=1)
                P.extend(logits.argmax(1).cpu().tolist())
                Y.extend(labels.cpu().tolist())
                S.extend(probs[:, 1].cpu().tolist())
            else:
                hp = hb_pred.detach().view(-1)
                if REG_CONFIG["normalize_hb"]: hp = _denorm_hb(hp)
                HBP.extend(hp.cpu().tolist()); HBT.extend(hb.cpu().view(-1).tolist())
    metrics = {"loss": total / max(len(loader), 1)}
    if TASK == "classification": metrics.update(cls_metrics(Y, P, S))
    else:                        metrics.update(reg_metrics(HBT, HBP))
    return metrics, Y, P, HBT, HBP, S

print("run_epoch_mm ready.")


## Section 9 — K-fold runner (test mean ± std)

For each fold: fit tabular stats on the fold's training rows, standardise train/val/**test**
with them (no leakage), build a fresh model via `model_factory(tab_dim)`, train with
early stopping, and evaluate on the **fixed** test set. Reports per-fold and mean±std —
the numbers to put in the paper. `K=1` → a single stratified holdout.

In [ ]:
RESULTS = []          # main-pass results
ABLATION_RESULTS = []  # ablation results

def _fold_indices():
    if K == 1:
        idx = np.arange(len(POOL_DF))
        tr, vl = train_test_split(idx, test_size=VAL_SPLIT,
                                  random_state=SEED, stratify=POOL_LBL)
        return [(tr, vl)]
    skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
    return list(skf.split(POOL_DF, POOL_LBL))

def _build_loaders(tr_df, vl_df, numeric_feats, use_ses, stats):
    T_TR, T_VA = build_transforms(True), build_transforms(False)
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=(DEVICE == "cuda"), drop_last=False)
    tr_ds = MMDataset(tr_df, T_TR, numeric_feats, use_ses, stats)
    vl_ld = DataLoader(MMDataset(vl_df, T_VA, numeric_feats, use_ses, stats),
                       shuffle=False, **kw)
    ts_ld = DataLoader(MMDataset(TEST_DF, T_VA, numeric_feats, use_ses, stats),
                       shuffle=False, **kw)
    if USE_AUGMENTATION:
        tr_ld = make_balanced_loader(
            dataset=tr_ds, hb_values=tr_df[HB_COL].values, batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS, device=DEVICE, hb_filter_min=HB_FILTER_MIN,
            hb_filter_max=HB_FILTER_MAX, aug_bins=AUG_BINS)
    else:
        tr_ld = DataLoader(tr_ds, shuffle=True, **kw)
    return tr_ld, vl_ld, ts_ld

def _make_sched(opt):
    if WARMUP_EPOCHS > 0 and EPOCHS > WARMUP_EPOCHS:
        from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
        w = LinearLR(opt, start_factor=0.05, end_factor=1.0, total_iters=WARMUP_EPOCHS)
        c = CosineAnnealingLR(opt, T_max=max(EPOCHS - WARMUP_EPOCHS, 1))
        return SequentialLR(opt, [w, c], milestones=[WARMUP_EPOCHS])
    if SCHEDULER == "cosine":  return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    if SCHEDULER == "plateau": return torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    return None

def run_model_kfold_mm(name, model_factory, numeric_feats, use_ses,
                       results_list=None, tag=""):
    """Train `name` with CV. model_factory(tab_dim)->model with forward(img,tab)."""
    if results_list is None: results_list = RESULTS
    tab_dim = len(numeric_feats) + (1 if use_ses else 0)
    folds = _fold_indices()
    sep = "=" * 72
    print(f"\n{sep}\n  {name}   [{('holdout' if K==1 else str(K)+'-fold')}]"
          f"   feats={numeric_feats}+{'SES' if use_ses else 'noSES'}  {tag}\n{sep}")

    accs, aucs, f1s, maes, rmses, r2s = [], [], [], [], [], []
    ts_scores, ts_preds, ts_hbp = [], [], []
    ts_y = ts_hbt = None
    t0 = time.time(); params = 0

    for fi, (tr_idx, vl_idx) in enumerate(folds):
        tr_df = POOL_DF.iloc[tr_idx]; vl_df = POOL_DF.iloc[vl_idx]
        stats = fit_tab_stats(tr_df, numeric_feats)
        tr_ld, vl_ld, ts_ld = _build_loaders(tr_df, vl_df, numeric_feats, use_ses, stats)

        cls_loss = build_cls_loss((tr_df[HB_COL] < HB_THRESH).astype(int).tolist())
        reg_loss = build_reg_loss()

        model = model_factory(tab_dim).to(DEVICE)
        if params == 0:
            params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        sch = _make_sched(opt)
        scaler = torch.cuda.amp.GradScaler() if USE_AMP and DEVICE == "cuda" else None
        stop = EarlyStopper(EARLY_STOP)

        for ep in range(1, EPOCHS + 1):
            tr, *_ = run_epoch_mm(model, tr_ld, cls_loss, reg_loss, opt, GRAD_ACCUM, scaler)
            vl, *_ = run_epoch_mm(model, vl_ld, cls_loss, reg_loss)
            if sch is not None:
                sch.step(vl["loss"]) if SCHEDULER == "plateau" else sch.step()
            line = f"    F{fi+1} Ep{ep:02d}/{EPOCHS} TrL {tr['loss']:.4f} VlL {vl['loss']:.4f}"
            if TASK == "classification":
                line += f"  Acc {vl['acc']:.3f} AUC {vl['auc']:.3f} F1 {vl['f1']:.3f}"
            else:
                line += f"  MAE {vl['mae']:.3f} RMSE {vl['rmse']:.3f}"
            print(line)
            if stop.step(vl["loss"]):
                print(f"    early stop @ ep{ep}"); break

        ts, y, p, hbt, hbp, sc = run_epoch_mm(model, ts_ld, cls_loss, reg_loss)
        if ts_y is None: ts_y, ts_hbt = y, hbt
        ts_scores.append(sc); ts_preds.append(p); ts_hbp.append(hbp)
        if TASK == "classification":
            accs.append(ts["acc"]); aucs.append(ts["auc"]); f1s.append(ts["f1"])
            print(f"    F{fi+1} TEST  Acc {ts['acc']:.4f}  AUC {ts['auc']:.4f}  F1 {ts['f1']:.4f}")
        else:
            maes.append(ts["mae"]); rmses.append(ts["rmse"]); r2s.append(ts["r2"])
            print(f"    F{fi+1} TEST  MAE {ts['mae']:.4f}  RMSE {ts['rmse']:.4f}  R2 {ts['r2']:.4f}")
        del model
        if DEVICE == "cuda": torch.cuda.empty_cache()
        gc.collect()

    elapsed = time.time() - t0
    print("  " + "-" * 68)
    row = dict(name=name, tag=tag, status="PASS", params=params, time_s=elapsed,
               folds=K, numeric_feats=list(numeric_feats), use_ses=use_ses)
    if TASK == "classification":
        row.update(acc_mean=float(np.nanmean(accs)), acc_std=float(np.nanstd(accs)),
                   auc_mean=float(np.nanmean(aucs)), auc_std=float(np.nanstd(aucs)),
                   f1_mean=float(np.nanmean(f1s)),  f1_std=float(np.nanstd(f1s)))
        print(f"  {name}:  Acc {row['acc_mean']:.4f}+/-{row['acc_std']:.4f}   "
              f"AUC {row['auc_mean']:.4f}+/-{row['auc_std']:.4f}   "
              f"F1 {row['f1_mean']:.4f}+/-{row['f1_std']:.4f}")
    else:
        row.update(mae_mean=float(np.nanmean(maes)), mae_std=float(np.nanstd(maes)),
                   rmse_mean=float(np.nanmean(rmses)), rmse_std=float(np.nanstd(rmses)),
                   r2_mean=float(np.nanmean(r2s)),   r2_std=float(np.nanstd(r2s)))
        print(f"  {name}:  MAE {row['mae_mean']:.4f}+/-{row['mae_std']:.4f}   "
              f"RMSE {row['rmse_mean']:.4f}+/-{row['rmse_std']:.4f}   "
              f"R2 {row['r2_mean']:.4f}+/-{row['r2_std']:.4f}")
    results_list.append(row)
    print(f"  done in {elapsed:.0f}s")
    return row

def _fail(name, e, results_list=None):
    if results_list is None: results_list = RESULTS
    print(f"  {name} FAILED: {e}"); traceback.print_exc()
    results_list.append(dict(name=name, status="FAIL", error=str(e)))

print("K-fold multimodal runner ready.")


## Section 10 — Model registry (all 17 image builders)

Each entry is a **zero-arg callable** returning a fresh image model (same build calls
as `run_ensemble.ipynb`). Modules are imported once and cached. Anything that fails to
import (e.g. a missing CUDA kernel) is simply skipped.

In [ ]:
def _add(folder):
    p = os.path.join(MODELS_DIR, folder)
    if p not in sys.path: sys.path.insert(0, p)

def _load(mod_name, folder, filename):
    _add(folder)
    spec = importlib.util.spec_from_file_location(
        mod_name, os.path.join(MODELS_DIR, folder, filename))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

def _register():
    reg = {}
    def safe(name, fn): reg[name] = fn
    # ---- Mamba variants ----
    try:
        m = _load("_m1", "01_Mamba_Official", "eye_hb_model.py")
        safe("Mamba1", lambda m=m: m.build_mamba1(IMG_SIZE, embed_dim=128, depth=4))
        safe("Mamba2", lambda m=m: m.build_mamba2(IMG_SIZE, embed_dim=128, depth=4))
    except Exception as e: print("  [skip] Mamba1/2:", e)
    try:
        m = _load("_m3", "06_Mamba3_Minimal", "eye_hb_model.py")
        safe("Mamba3", lambda m=m: m.build_model(IMG_SIZE, embed_dim=128, depth=4))
    except Exception as e: print("  [skip] Mamba3:", e)
    for key, folder in [("VMamba", "02_VMamba"), ("MambaVision", "03_MambaVision"),
                        ("MedMamba", "04_MedMamba"), ("VSSD", "05_VSSD_Mamba2Vision"),
                        ("DSA_Mamba", "07_DSA_Mamba_Custom")]:
        try:
            m = _load("_" + key, folder, "eye_hb_model.py")
            safe(key, lambda m=m: m.build_model(IMG_SIZE))
        except Exception as e: print(f"  [skip] {key}:", e)
    try:
        m = _load("_em", "08_EfficientMamba", "eye_hb_model.py")
        safe("EfficientMamba", lambda m=m: m.build_model(
            img_size=IMG_SIZE, embed_dim=256, depth=4, freeze_backbone=FREEZE_BACKBONE))
    except Exception as e: print("  [skip] EfficientMamba:", e)
    try:
        m = _load("_asm", "09_AdaptiveScanMamba", "adaptive_scan_mamba.py")
        safe("AdaptiveScan", lambda m=m: m.build_adaptive_scan_mamba(
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], d_state=NEW_MODEL_CFG["d_state"],
            colour_proj=NEW_MODEL_CFG["colour_proj"]))
    except Exception as e: print("  [skip] AdaptiveScan:", e)
    try:
        m = _load("_moe", "10_MoEMamba", "moe_mamba.py")
        safe("MoEMamba", lambda m=m: m.build_moe_mamba(
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], n_experts=NEW_MODEL_CFG["n_experts"],
            d_state=NEW_MODEL_CFG["d_state"]))
    except Exception as e: print("  [skip] MoEMamba:", e)
    try:
        m = _load("_cf", "11_CrossFusionMamba", "cross_fusion_mamba.py")
        safe("CrossFusion", lambda m=m: m.build_cross_fusion_mamba(
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], d_state=NEW_MODEL_CFG["d_state"]))
    except Exception as e: print("  [skip] CrossFusion:", e)
    # ---- CNN / ViT baselines (timm) ----
    try:
        import timm
        class TimmWrapper(nn.Module):
            def __init__(self, backbone_name, pretrained=True):
                super().__init__()
                self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
                fd = self.backbone.num_features
                self.cls_head = nn.Sequential(nn.LayerNorm(fd), nn.Linear(fd, 2))
                self.reg_head = nn.Sequential(nn.LayerNorm(fd), nn.Linear(fd, 1))
            def forward(self, x):
                f = self.backbone(x)
                return self.cls_head(f), self.reg_head(f).squeeze(-1)
        for key, bb in [("ResNet50", "resnet50"), ("EfficientNetB3", "efficientnet_b3"),
                        ("DenseNet121", "densenet121"), ("MobileNetV3", "mobilenetv3_large_100"),
                        ("ViT_B16", "vit_base_patch16_224")]:
            safe(key, lambda bb=bb: TimmWrapper(bb, pretrained=True))
    except Exception as e: print("  [skip] timm baselines:", e)
    return reg

IMG_BUILDERS = _register()
print("Registered image builders:", list(IMG_BUILDERS.keys()))


## Section 11 — MAIN PASS: all models with the default fusion

Runs every enabled model. If `USE_TABULAR=True`, each backbone is wrapped in
`MultiModalWrapper(..., fusion=FUSION)` with the configured tabular features; otherwise
`ImageOnlyWrapper` (pure image baseline). Same K-fold protocol for all.

In [ ]:
def make_factory(img_build_fn, use_tabular, fusion):
    if use_tabular:
        return lambda tab_dim: MultiModalWrapper(img_build_fn(), tab_dim, fusion=fusion)
    return lambda tab_dim: ImageOnlyWrapper(img_build_fn())

_active_numeric = NUMERIC_FEATURES if USE_TABULAR else []
_active_ses     = USE_SES if USE_TABULAR else False

for name, on in RUN.items():
    if not on:
        print(f"{name} skipped."); continue
    if name not in IMG_BUILDERS:
        print(f"{name}: no builder available, skipped."); continue
    try:
        factory = make_factory(IMG_BUILDERS[name], USE_TABULAR, FUSION)
        run_model_kfold_mm(name, factory, _active_numeric, _active_ses,
                           tag=(f"fusion={FUSION}" if USE_TABULAR else "image-only"))
    except Exception as e:
        _fail(name, e)


## Section 12 — Main results table & chart

In [ ]:
def results_frame(rows):
    ok = [r for r in rows if r.get("status") == "PASS"]
    if not ok:
        print("No successful runs."); return None
    df = pd.DataFrame(ok)
    if TASK == "classification":
        df = df.sort_values("auc_mean", ascending=False)
        cols = ["name", "tag", "params", "acc_mean", "acc_std",
                "auc_mean", "auc_std", "f1_mean", "f1_std", "time_s"]
    else:
        df = df.sort_values("mae_mean", ascending=True)
        cols = ["name", "tag", "params", "mae_mean", "mae_std",
                "rmse_mean", "rmse_std", "r2_mean", "r2_std", "time_s"]
    return df[[c for c in cols if c in df.columns]].reset_index(drop=True)

MAIN_DF = results_frame(RESULTS)
if MAIN_DF is not None:
    print(MAIN_DF.to_string(index=False))
    MAIN_DF.to_csv("multimamba_main_results.csv", index=False)

    fig, ax = plt.subplots(figsize=(11, max(3, 0.45 * len(MAIN_DF))))
    if TASK == "classification":
        ax.barh(MAIN_DF["name"], MAIN_DF["auc_mean"],
                xerr=MAIN_DF["auc_std"], color="steelblue")
        ax.set_xlabel("Test AUC (mean +/- std)")
    else:
        ax.barh(MAIN_DF["name"], MAIN_DF["mae_mean"],
                xerr=MAIN_DF["mae_std"], color="indianred")
        ax.set_xlabel("Test MAE g/dL (lower is better)")
    ax.invert_yaxis(); ax.set_title(f"MULTIMAMBA — main pass ({CV_DESC}, fusion={FUSION})")
    plt.tight_layout(); plt.savefig("multimamba_main_results.png", dpi=110,
                                    bbox_inches="tight"); plt.show()


# ─────────────────────────  ABLATIONS  ─────────────────────────

Everything below is diagnostic and runs on a single backbone (`ABLATION_MODEL`) so the
comparisons are apples-to-apples. Three studies:

1. **Fusion sweep** — which of `concat / film / gate / attn` fuses the tabular data best.
2. **Per-variable contribution** — image-only → +Height → +Weight → +SES, so you can
   see exactly how much each new variable helps.
3. **Tabular-only reference** — how much signal is in the clinical variables alone.

## Ablation 1 — Fusion-mode sweep

In [ ]:
FUSION_ABL = []
if RUN_ABLATION_FUSION:
    if ABLATION_MODEL not in IMG_BUILDERS:
        print(f"ABLATION_MODEL '{ABLATION_MODEL}' has no builder; skipping fusion sweep.")
    else:
        for mode in FUSION_MODES:
            try:
                fac = (lambda mode: (lambda tab_dim: MultiModalWrapper(
                    IMG_BUILDERS[ABLATION_MODEL](), tab_dim, fusion=mode)))(mode)
                run_model_kfold_mm(f"{ABLATION_MODEL} [{mode}]", fac,
                                   NUMERIC_FEATURES, USE_SES,
                                   results_list=FUSION_ABL, tag=f"fusion={mode}")
            except Exception as e:
                _fail(f"{ABLATION_MODEL} [{mode}]", e, results_list=FUSION_ABL)
    fdf = results_frame(FUSION_ABL)
    if fdf is not None:
        print("\nFusion comparison:\n", fdf.to_string(index=False))
        fdf.to_csv("ablation_fusion.csv", index=False)
else:
    print("RUN_ABLATION_FUSION = False -> skipped.")


## Ablation 2 — Per-variable contribution (image-only → +Height → +Weight → +SES)

Uses the default `FUSION`. The first step (image-only) uses `ImageOnlyWrapper`; the rest
add one variable at a time so the marginal effect of each is visible.

In [ ]:
VAR_ABL = []
if RUN_ABLATION_VARIABLE:
    if ABLATION_MODEL not in IMG_BUILDERS:
        print(f"ABLATION_MODEL '{ABLATION_MODEL}' has no builder; skipping variable ablation.")
    else:
        for label, num_feats, use_ses in ABLATION_STEPS:
            tab_dim = len(num_feats) + (1 if use_ses else 0)
            try:
                if tab_dim == 0:
                    fac = lambda tab_dim: ImageOnlyWrapper(IMG_BUILDERS[ABLATION_MODEL]())
                else:
                    fac = lambda tab_dim: MultiModalWrapper(
                        IMG_BUILDERS[ABLATION_MODEL](), tab_dim, fusion=FUSION)
                run_model_kfold_mm(f"{ABLATION_MODEL} [{label}]", fac,
                                   num_feats, use_ses,
                                   results_list=VAR_ABL, tag=label)
            except Exception as e:
                _fail(f"{ABLATION_MODEL} [{label}]", e, results_list=VAR_ABL)
    vdf = results_frame(VAR_ABL)
    if vdf is not None:
        print("\nPer-variable contribution:\n", vdf.to_string(index=False))
        vdf.to_csv("ablation_variables.csv", index=False)
        ok = [r for r in VAR_ABL if r.get("status") == "PASS"]
        if ok:
            labels = [r["tag"] for r in ok]
            fig, ax = plt.subplots(figsize=(8, 4))
            if TASK == "classification":
                y = [r["auc_mean"] for r in ok]; e = [r["auc_std"] for r in ok]
                ax.set_ylabel("Test AUC")
            else:
                y = [r["mae_mean"] for r in ok]; e = [r["mae_std"] for r in ok]
                ax.set_ylabel("Test MAE g/dL (lower better)")
            ax.errorbar(range(len(y)), y, yerr=e, marker="o", capsize=4)
            ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=20, ha="right")
            ax.set_title(f"{ABLATION_MODEL}: adding variables (fusion={FUSION})")
            plt.tight_layout(); plt.savefig("ablation_variables.png", dpi=110,
                                            bbox_inches="tight"); plt.show()
else:
    print("RUN_ABLATION_VARIABLE = False -> skipped.")


## Ablation 3 — Tabular-only reference (no image)

In [ ]:
TAB_ABL = []
if RUN_ABLATION_TABONLY:
    try:
        fac = lambda tab_dim: TabularOnlyModel(tab_dim, hidden=TAB_HIDDEN)
        run_model_kfold_mm("Tabular-only MLP", fac, NUMERIC_FEATURES, USE_SES,
                           results_list=TAB_ABL, tag="tabular-only")
    except Exception as e:
        _fail("Tabular-only MLP", e, results_list=TAB_ABL)
    tdf = results_frame(TAB_ABL)
    if tdf is not None:
        print("\nTabular-only:\n", tdf.to_string(index=False))
        tdf.to_csv("ablation_tabular_only.csv", index=False)
else:
    print("RUN_ABLATION_TABONLY = False -> skipped.")


## Section 13 — Combined summary

In [ ]:
all_rows = RESULTS + FUSION_ABL + VAR_ABL + TAB_ABL
summary = results_frame([r for r in all_rows if r.get("status") == "PASS"])
if summary is not None:
    print("=" * 80)
    print("COMBINED SUMMARY  (main pass + ablations)")
    print("=" * 80)
    print(summary.to_string(index=False))
    summary.to_csv("multimamba_summary.csv", index=False)
    print("\nSaved CSVs: multimamba_main_results.csv, ablation_*.csv, multimamba_summary.csv")
else:
    print("Nothing to summarise yet — run the model cells above.")
